# Urban Heat & Cooling-Priority Mapping — Adaptive Capacity Pillar (Greenery)

**NUS-ISS Practice Module, Week 2.** Builds `adaptive_capacity_pillar.csv` for
`rank_impact.ipynb` as an **interim NDVI-threshold proxy** for vegetation
fraction — no manual labeling required. The real version should eventually
come from Track B's S3 land-cover output (U-Net+RF ensemble, validated on
the 300 hand-labeled points); this fills the gap until that's ready. One
notebook, run top to bottom:

1. **Setup** — install deps, authenticate Earth Engine, mount Drive
2. **AC.1** — config (must match `gee_heat_variants.ipynb`'s season window)
3. **AC.2** — fetch URA subzones from data.gov.sg (same as `gee_heat_variants.ipynb` S2.2)
4. **AC.3** — build the season-controlled Sentinel-2 NDVI composite
5. **AC.4** — threshold NDVI into a vegetation mask (⚠️ placeholder cutoff — see AC.4)
6. **AC.5** — zonal vegetation fraction per subzone
7. **AC.6** — match subzone_id to heat-variants CSV
8. **AC.7** — verdict
9. **AC.8** — save to Drive

Run cells in order.

---


# SETUP — run once per session

## Setup 1 — Install dependencies

In [1]:
# --- SETUP CELL 1: Install all dependencies used in this notebook ----------
!pip install -q earthengine-api pandas requests


## Setup 2 — Authenticate & initialize Earth Engine

In [2]:
# --- SETUP CELL 2: Authenticate & initialize Earth Engine -------------------
import ee

PROJECT_ID = "nus-iss-urban-heat-sg"  # <-- your GCP project

if PROJECT_ID == "your-gcp-project-id":
    raise ValueError(
        "PROJECT_ID is still the placeholder. Set it to your actual GCP project ID, "
        "then re-run this cell."
    )

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("EE initialized OK, project:", PROJECT_ID)


EE initialized OK, project: nus-iss-urban-heat-sg


## Setup 3 — Mount Google Drive

In [3]:
# --- SETUP CELL 3: Mount Google Drive ----------------------------------------
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


Mounted at /content/drive
Drive mounted at /content/drive


## Setup 4 — Initialize shared results tracker

In [4]:
# --- SETUP CELL 4: Initialize shared results tracker ------------------------
ac_results = {}
print("ac_results initialized — populated by the AC.7 verdict cell.")


ac_results initialized — populated by the AC.7 verdict cell.


---
# AC — Adaptive Capacity Pillar (interim NDVI-threshold greenery proxy)


## AC.1 — Config

`YEARS` / `DRY_SEASON_MONTHS` must match `gee_heat_variants.ipynb` exactly —
a different season window here would make this pillar's vegetation snapshot
inconsistent with the heat-variant composites it's being combined with.


In [5]:
# --- AC CELL 1: Config -------------------------------------------------------
sg_bbox = ee.Geometry.Rectangle([103.55, 1.15, 104.10, 1.48])

# Must match gee_heat_variants.ipynb S2.1 exactly.
YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
DRY_SEASON_MONTHS = [4, 5, 10, 11]  # Both inter-monsoon periods — locked C4 decision,
                                     # confirmed via season_window_diagnostic.ipynb

S2_CLOUD_PROB_MAX = 40
TARGET_SCALE = 10

NDVI_VEGETATION_THRESHOLD = 0.35  # <-- placeholder, same cutoff as Week-1 G3.10's toy land cover

EXPORT_FOLDER = "urban_heat_sg"
HEAT_CSV_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/heat_variants_subzone.csv"
OUT_PATH = f"/content/drive/MyDrive/{EXPORT_FOLDER}/adaptive_capacity_pillar.csv"

SUBZONE_ID_PROPERTY = "SUBZONE_N"

import os
if not os.path.exists(HEAT_CSV_PATH):
    raise FileNotFoundError(
        f"{HEAT_CSV_PATH} not found. Run gee_heat_variants.ipynb's export first, "
        f"then re-run this cell."
    )
print("✅ Heat-variants CSV found — will match subzone_id against it in AC.6.")
print(f"Season window: months {DRY_SEASON_MONTHS} across years {YEARS} — confirm this matches gee_heat_variants.ipynb.")


✅ Heat-variants CSV found — will match subzone_id against it in AC.6.
Season window: months [4, 5, 10, 11] across years [2021, 2022, 2023, 2024, 2025, 2026] — confirm this matches gee_heat_variants.ipynb.


## AC.2 — Fetch URA subzones directly from data.gov.sg

Same fetch + property-sanitization pattern as `gee_heat_variants.ipynb` S2.2
— this is a separate Colab session, so subzones need to be re-fetched rather
than reused from that notebook's memory.


In [6]:
# --- AC CELL 2: Fetch URA subzones from data.gov.sg --------------------------
import requests
import json

SUBZONE_DATASET_ID = "d_8594ae9ff96d0c708bc2af633048edfb"  # MP19 Subzone Boundary (No Sea), GEOJSON
SUBZONE_LOCAL_PATH = "/content/ura_subzones.geojson"

def fetch_datagovsg_geojson(dataset_id, out_path):
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    r = requests.get(poll_url)
    r.raise_for_status()
    payload = r.json()
    if payload.get("code") != 0:
        raise RuntimeError(f"data.gov.sg API error: {payload.get('errMsg')}")
    download_url = payload["data"]["url"]
    geojson_bytes = requests.get(download_url).content
    with open(out_path, "wb") as f:
        f.write(geojson_bytes)
    return out_path

subzone_path = fetch_datagovsg_geojson(SUBZONE_DATASET_ID, SUBZONE_LOCAL_PATH)
print(f"Downloaded subzones GeoJSON -> {subzone_path}")

with open(subzone_path) as f:
    subzone_geojson = json.load(f)

n_features_local = len(subzone_geojson.get("features", []))
if n_features_local == 0:
    raise ValueError("Downloaded GeoJSON has zero features — check the download before proceeding.")

renamed_props = set()
for feature in subzone_geojson.get("features", []):
    props = feature.get("properties", {})
    for old_key in list(props.keys()):
        if "." in old_key:
            new_key = old_key.replace(".", "_")
            props[new_key] = props.pop(old_key)
            renamed_props.add(f"{old_key} -> {new_key}")
if renamed_props:
    print(f"Sanitized {len(renamed_props)} property name(s) containing '.': {sorted(renamed_props)}")

subzones = ee.FeatureCollection(subzone_geojson)
n_subzones_total = subzones.size().getInfo()
print(f"Loaded {n_subzones_total} subzones into an ee.FeatureCollection")

sample_props = subzones.first().propertyNames().getInfo()
if SUBZONE_ID_PROPERTY not in sample_props:
    raise ValueError(f"SUBZONE_ID_PROPERTY '{SUBZONE_ID_PROPERTY}' not found. Candidates: {sample_props}")
print(f"✅ SUBZONE_ID_PROPERTY = '{SUBZONE_ID_PROPERTY}' confirmed present.")


Downloaded subzones GeoJSON -> /content/ura_subzones.geojson
Sanitized 2 property name(s) containing '.': ['SHAPE.AREA -> SHAPE_AREA', 'SHAPE.LEN -> SHAPE_LEN']
Loaded 332 subzones into an ee.FeatureCollection
✅ SUBZONE_ID_PROPERTY = 'SUBZONE_N' confirmed present.


## AC.3 — Season-controlled Sentinel-2 NDVI composite

In [7]:
# --- AC CELL 3: Sentinel-2 NDVI composite (season-controlled, matches C4) --
def date_filter_for_years_months(collection, years, months):
    filters = []
    for y in years:
        for m in months:
            start = ee.Date.fromYMD(y, m, 1)
            end = start.advance(1, "month")
            filters.append(ee.Filter.date(start, end))
    return collection.filter(ee.Filter.Or(*filters))


def mask_s2_clouds(cloud_prob_image):
    return cloud_prob_image.select("probability").lt(S2_CLOUD_PROB_MAX)


s2_sr = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(sg_bbox)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
)
s2_sr = date_filter_for_years_months(s2_sr, YEARS, DRY_SEASON_MONTHS)

s2_cloud_prob = ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY").filterBounds(sg_bbox)
s2_cloud_prob = date_filter_for_years_months(s2_cloud_prob, YEARS, DRY_SEASON_MONTHS)

joined = ee.Join.saveFirst("cloud_mask").apply(
    primary=s2_sr,
    secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField="system:index", rightField="system:index"),
)

def _mask_and_ndvi(img):
    img = ee.Image(img)
    cloud_img = ee.Image(img.get("cloud_mask"))
    clear_mask = mask_s2_clouds(cloud_img)
    img = img.updateMask(clear_mask)
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return img.addBands(ndvi)

s2_indexed = ee.ImageCollection(joined).map(_mask_and_ndvi)
ndvi_composite = s2_indexed.select("NDVI").median().clip(sg_bbox)

print("NDVI composite built.")


NDVI composite built.


## AC.4 — Threshold into a vegetation mask

⚠️ `NDVI_VEGETATION_THRESHOLD = 0.35` (set in AC.1) is a placeholder, not a
validated cutoff — it's the same value your Week-1 G3.10 toy land cover used
for a quick wiring test on one tile, not something tuned against ground
truth. Once Track B's real S3 land-cover classifier exists, replace this
pillar with actual vegetation-class fraction instead of an NDVI threshold.


In [8]:
# --- AC CELL 4: Vegetation mask -----------------------------------------------
vegetation_mask = ndvi_composite.gt(NDVI_VEGETATION_THRESHOLD).rename("is_vegetation")
print(f"Vegetation mask built (NDVI > {NDVI_VEGETATION_THRESHOLD}).")


Vegetation mask built (NDVI > 0.35).


## AC.5 — Zonal vegetation fraction per subzone

In [9]:
# --- AC CELL 5: Zonal vegetation fraction -------------------------------------
zonal = vegetation_mask.reduceRegions(
    collection=subzones,
    reducer=ee.Reducer.mean(),  # mean of a 0/1 mask = fraction of vegetated pixels
    scale=TARGET_SCALE,
    tileScale=4,
)

n_zonal = zonal.size().getInfo()
n_null = zonal.filter(ee.Filter.eq("mean", None)).size().getInfo()
print(f"Zonal reduction: {n_zonal} subzones, {n_null} with null mean (no valid pixels).")
if n_null:
    print("⚠️  Non-zero nulls expected for tiny/sliver subzones — confirm this count is small, not most of them.")

records = zonal.map(
    lambda f: ee.Feature(None, {
        "subzone_id": f.get(SUBZONE_ID_PROPERTY),
        "greenery_fraction": f.get("mean"),
    })
).getInfo()

import pandas as pd
ac_df = pd.DataFrame([r["properties"] for r in records["features"]])
print(f"\nPulled {len(ac_df)} rows client-side.")
ac_df.head(10)


Zonal reduction: 332 subzones, 0 with null mean (no valid pixels).

Pulled 332 rows client-side.


,greenery_fraction,subzone_id
0,0.525886,DEPOT ROAD
1,0.420562,BUKIT MERAH
2,0.132039,CHINATOWN
3,0.131398,PHILLIP
4,0.026145,RAFFLES PLACE
5,0.034570,CHINA SQUARE
6,0.403443,TIONG BAHRU
7,0.125238,BAYFRONT SUBZONE
8,0.487375,TIONG BAHRU STATION
9,0.020159,CLIFFORD PIER


## AC.6 — Match subzone_id to heat-variants CSV

Since both this notebook and `gee_heat_variants.ipynb` derive `subzone_id`
from the same `SUBZONE_N` property on the same GeoJSON, this should match
exactly (no case-normalization needed, unlike SP.5's SingStat name match)
— but checked explicitly rather than assumed.


In [10]:
# --- AC CELL 6: Match subzone_id ----------------------------------------------
heat = pd.read_csv(HEAT_CSV_PATH)
heat_ids = set(heat["subzone_id"].astype(str))
ac_ids = set(ac_df["subzone_id"].astype(str))

n_matched = len(heat_ids & ac_ids)
n_unmatched_heat = len(heat_ids - ac_ids)
n_unmatched_ac = len(ac_ids - heat_ids)

print(f"Matched: {n_matched}")
print(f"Heat-CSV subzones with no greenery match: {n_unmatched_heat}")
print(f"Greenery subzones with no heat-CSV match: {n_unmatched_ac}")

if n_unmatched_heat:
    print("\n⚠️  Unexpected — both files should derive subzone_id from the same GeoJSON property.")
    print("   Sample mismatches:", sorted(heat_ids - ac_ids)[:10])
else:
    print("\n✅ All heat-CSV subzones have a greenery match, as expected.")

ac_df = ac_df[ac_df["subzone_id"].astype(str).isin(heat_ids)].copy()


Matched: 332
Heat-CSV subzones with no greenery match: 0
Greenery subzones with no heat-CSV match: 0

✅ All heat-CSV subzones have a greenery match, as expected.


## AC.7 — Verdict

In [11]:
# --- AC CELL 7: Verdict --------------------------------------------------------
print("\n--- AC Verdict ---")

ac_checks = {
    "NDVI composite built (non-error)": ndvi_composite is not None,
    "Zonal reduction retained subzones (non-zero)": n_zonal > 0,
    "No majority null in zonal reduction": n_null < n_zonal * 0.5 if n_zonal else False,
    "Majority of heat-CSV subzones matched (>=90%)": n_matched >= 0.9 * len(heat_ids),
    "greenery_fraction present for all matched rows": ac_df["greenery_fraction"].notna().all(),
}

for check, passed in ac_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {check}")

ac_pass = all(ac_checks.values())
print("\n⚠️  Reminder: this is an NDVI-threshold proxy, not Track B's real S3 land cover. "
      "Fine to unblock TOY_MODE=False testing now; revisit once S3 is ready.")
if ac_pass:
    print("✅ AC PASS: adaptive_capacity_pillar.csv ready to save.")
else:
    print("⚠️  AC FAIL/MARGINAL: resolve flagged step(s) above first.")

ac_results["AC_adaptive_capacity_pillar"] = {
    "status": "PASS" if ac_pass else "FAIL",
    "checks": ac_checks,
    "n_subzones_matched": n_matched,
    "ndvi_threshold": NDVI_VEGETATION_THRESHOLD,
}



--- AC Verdict ---
  [PASS] NDVI composite built (non-error)
  [PASS] Zonal reduction retained subzones (non-zero)
  [PASS] No majority null in zonal reduction
  [PASS] Majority of heat-CSV subzones matched (>=90%)
  [PASS] greenery_fraction present for all matched rows

⚠️  Reminder: this is an NDVI-threshold proxy, not Track B's real S3 land cover. Fine to unblock TOY_MODE=False testing now; revisit once S3 is ready.
✅ AC PASS: adaptive_capacity_pillar.csv ready to save.


## AC.8 — Save to Drive

In [12]:
# --- AC CELL 8: Save to Drive ---------------------------------------------------
out = ac_df[["subzone_id", "greenery_fraction"]]
out.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH} ({len(out)} subzones)")
print("\nPoint ADAPTIVE_CSV_PATH at this file in rank_impact.ipynb's RI.1.")
print("Once both sensitivity_pillar.csv and adaptive_capacity_pillar.csv exist, set TOY_MODE = False there.")


Saved: /content/drive/MyDrive/urban_heat_sg/adaptive_capacity_pillar.csv (332 subzones)

Point ADAPTIVE_CSV_PATH at this file in rank_impact.ipynb's RI.1.
Once both sensitivity_pillar.csv and adaptive_capacity_pillar.csv exist, set TOY_MODE = False there.
